# Whisper Fine-Tuning on Common Voice (Telugu) with HuggingFace Trainer

In [ ]:
!pip install -q datasets transformers evaluate jiwer accelerate librosa soundfile

In [ ]:
import torchimport evaluatefrom dataclasses import dataclassfrom typing import Any, Dict, List, Unionfrom datasets import load_dataset, DatasetDict, Audiofrom transformers import (    WhisperFeatureExtractor,    WhisperTokenizer,    WhisperProcessor,    WhisperForConditionalGeneration,    Seq2SeqTrainingArguments,    Seq2SeqTrainer,)

In [ ]:
DATASET_PATH = ""DATASET_CONFIG = "te"MODEL_NAME = "openai/whisper-small"LANGUAGE = "telugu"TASK = "transcribe"SAMPLING_RATE = 16000OUTPUT_DIR = "./whisper-small-te"device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
common_voice = DatasetDict()common_voice["train"] = load_dataset(DATASET_PATH, DATASET_CONFIG, split="train+validation")common_voice["test"] = load_dataset(DATASET_PATH, DATASET_CONFIG, split="test")common_voice = common_voice.remove_columns([    "accent", "age", "client_id", "down_votes", "gender",    "locale", "segment", "up_votes", "path"])print(common_voice)

In [ ]:
feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)tokenizer = WhisperTokenizer.from_pretrained(MODEL_NAME, language=LANGUAGE, task=TASK)processor = WhisperProcessor.from_pretrained(MODEL_NAME, language=LANGUAGE, task=TASK)

In [ ]:
common_voice = common_voice.cast_column("audio", Audio(sampling_rate=SAMPLING_RATE))

In [ ]:
def prepare_dataset(batch):    audio = batch["audio"]    batch["input_features"] = feature_extractor(        audio["array"], sampling_rate=audio["sampling_rate"]    ).input_features[0]    batch["labels"] = tokenizer(batch["sentence"]).input_ids    return batch

In [ ]:
common_voice = common_voice.map(    prepare_dataset,    remove_columns=common_voice.column_names["train"],    num_proc=1,)

In [ ]:
@dataclassclass DataCollatorSpeechSeq2SeqWithPadding:    processor: Any    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:        input_features = [{"input_features": feature["input_features"]} for feature in features]        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")        label_features = [{"input_ids": feature["labels"]} for feature in features]        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():            labels = labels[:, 1:]        batch["labels"] = labels        return batch

In [ ]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [ ]:
wer_metric = evaluate.load("wer")def compute_metrics(pred):    pred_ids = pred.predictions    label_ids = pred.label_ids    label_ids[label_ids == -100] = tokenizer.pad_token_id    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)    return {"wer": wer}

In [ ]:
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)model.generation_config.language = LANGUAGEmodel.generation_config.task = TASKmodel.generation_config.forced_decoder_ids = Nonemodel.config.use_cache = Falsemodel = model.to(device)

In [ ]:
training_args = Seq2SeqTrainingArguments(    output_dir=OUTPUT_DIR,    per_device_train_batch_size=16,    gradient_accumulation_steps=1,    learning_rate=1e-5,    warmup_steps=500,    max_steps=4000,    gradient_checkpointing=True,    fp16=torch.cuda.is_available(),    eval_strategy="steps",    per_device_eval_batch_size=8,    predict_with_generate=True,    generation_max_length=225,    save_steps=1000,    eval_steps=1000,    logging_steps=25,    report_to=["tensorboard"],    load_best_model_at_end=True,    metric_for_best_model="wer",    greater_is_better=False,    push_to_hub=False,)

In [ ]:
trainer = Seq2SeqTrainer(    args=training_args,    model=model,    train_dataset=common_voice["train"],    eval_dataset=common_voice["test"],    data_collator=data_collator,    compute_metrics=compute_metrics,    tokenizer=processor.feature_extractor,)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model(OUTPUT_DIR)processor.save_pretrained(OUTPUT_DIR)

In [ ]:
metrics = trainer.evaluate()print(metrics)